<a href="https://colab.research.google.com/github/irfanmohamed07/Agent-Atlas---Youtube-summarizer/blob/main/nano_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# PyTorch is the library we use to build and train our neural network.
import torch
import torch.nn as nn
import torch.nn.functional as F

# Use GPU if Colab gives us one.
# GPU makes neural-network calculations much faster.
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Running on:", device)

Running on: cpu


In [ ]:
# This is the ONLY knowledge our tiny model will learn from.
text = """
mohamed irfan is a good boy, irfan was born in 2005. he like ice cream
"""

print(text)


mohamed irfan is a good boy, irfan was born in 2005. he like ice cream



In [ ]:
# Get every unique character in our dataset.
# Example: m, o, h, a, space, i, r, f, n, ...
chars = sorted(list(set(text)))

# Number of unique characters.
# This is our vocabulary size.
vocab_size = len(chars)

# Character → number
# Example:
# "a" → 0
# "b" → 1
# "c" → 2
# ...
stoi = {
    ch: i
    for i, ch in enumerate(chars)
}

# Number → character
# This lets us convert the model's output back into text.
itos = {
    i: ch
    for i, ch in enumerate(chars)
}


# Convert text → numbers
def encode(text):
    return [stoi[c] for c in text]


# Convert numbers → text
def decode(tokens):
    return "".join(itos[i] for i in tokens)


encoded = encode(text)

print("Characters:")
print(chars)

print("\nVocabulary size:")
print(vocab_size)

print("\nFirst 30 token IDs:")
print(encoded[:30])

Characters:
['\n', ' ', ',', '.', '0', '2', '5', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'r', 's', 'w', 'y']

Vocabulary size:
25

First 30 token IDs:
[0, 18, 20, 14, 7, 18, 11, 10, 1, 15, 21, 12, 7, 19, 1, 15, 22, 1, 7, 1, 13, 20, 20, 10, 1, 8, 20, 24, 2, 1]


In [ ]:
# Neural networks work with tensors.
# So we convert our list of token IDs into a PyTorch tensor.

data = torch.tensor(
    encoded,
    dtype=torch.long
)

print(data)
print(data.shape)

tensor([ 0, 18, 20, 14,  7, 18, 11, 10,  1, 15, 21, 12,  7, 19,  1, 15, 22,  1,
         7,  1, 13, 20, 20, 10,  1,  8, 20, 24,  2,  1, 15, 21, 12,  7, 19,  1,
        23,  7, 22,  1,  8, 20, 21, 19,  1, 15, 19,  1,  5,  4,  4,  6,  3,  1,
        14, 11,  1, 17, 15, 16, 11,  1, 15,  9, 11,  1,  9, 21, 11,  7, 18,  0])
torch.Size([72])


In [ ]:
# The model learns to predict the NEXT character.

# Example:
#
# X: m o h a m
# Y: o h a m e
#
# So:
# m → o
# o → h
# h → a
# a → m
# m → e

block_size = 32

X = data[:block_size]

Y = data[1:block_size + 1]

print("X:")
print(decode(X.tolist()))

print("\nY:")
print(decode(Y.tolist()))

X:

mohamed irfan is a good boy, ir

Y:
mohamed irfan is a good boy, irf


In [ ]:
BATCH_SIZE = 16

def get_batch():

    # Pick random starting positions.
    ix = torch.randint(
        len(data) - block_size,
        (BATCH_SIZE,)
    )

    # Create input sequences.
    X = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    # Create target sequences.
    # Target is always one token ahead.
    Y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    return X.to(device), Y.to(device)


X, Y = get_batch()

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: torch.Size([16, 32])
Y shape: torch.Size([16, 32])


In [ ]:
N_EMBD = 64

# Every token gets a vector containing 64 numbers.
#
# If:
# "m" = token ID 20
#
# embedding might become:
#
# [0.2, -0.5, 0.8, ...]
#
# These numbers are LEARNED during training.

token_embedding = nn.Embedding(
    vocab_size,
    N_EMBD
).to(device)


token_vectors = token_embedding(X)

print("Token IDs shape:")
print(X.shape)

print("\nToken vectors shape:")
print(token_vectors.shape)

Token IDs shape:
torch.Size([16, 32])

Token vectors shape:
torch.Size([16, 32, 64])


In [ ]:
# Create a vector for every possible position.
#
# Position 0 → vector
# Position 1 → vector
# Position 2 → vector
# ...

position_embedding = nn.Embedding(
    block_size,
    N_EMBD
).to(device)


# Create positions:
positions = torch.arange(
    block_size,
    device=device
)

position_vectors = position_embedding(
    positions
)

print(position_vectors.shape)

torch.Size([32, 64])


In [ ]:
# Token information
#
# +
#
# Position information
#
# =
#
# Information that enters the Transformer

x = token_vectors + position_vectors

print("Transformer input shape:")
print(x.shape)

Transformer input shape:
torch.Size([16, 32, 64])


In [ ]:
head_size = N_EMBD // 4

# These three Linear layers learn how to create:
#
# Query = What am I looking for?
# Key   = What information do I have?
# Value = What information can I provide?

key = nn.Linear(
    N_EMBD,
    head_size,
    bias=False
).to(device)

query = nn.Linear(
    N_EMBD,
    head_size,
    bias=False
).to(device)

value = nn.Linear(
    N_EMBD,
    head_size,
    bias=False
).to(device)


K = key(x)
Q = query(x)
V = value(x)

print("K:", K.shape)
print("Q:", Q.shape)
print("V:", V.shape)

K: torch.Size([16, 32, 16])
Q: torch.Size([16, 32, 16])
V: torch.Size([16, 32, 16])


In [ ]:
# Compare every Query with every Key.
#
# This tells us:
#
# "How relevant is token B to token A?"

attention_scores = Q @ K.transpose(
    -2,
    -1
)

# Scale the scores.
# This prevents the values from becoming too large.

attention_scores = attention_scores / (
    head_size ** 0.5
)

print(attention_scores.shape)

torch.Size([16, 32, 32])


In [ ]:
# Create a lower-triangular matrix.
#
# Example:
#
# 1 0 0 0
# 1 1 0 0
# 1 1 1 0
# 1 1 1 1
#
# A token can see itself and previous tokens,
# but NOT future tokens.

mask = torch.tril(
    torch.ones(
        block_size,
        block_size,
        device=device
    )
)


attention_scores = attention_scores.masked_fill(
    mask == 0,
    float("-inf")
)

In [ ]:
# Convert attention scores into probabilities.
#
# Example:
#
# [2.0, 1.0, 0.5]
#
# becomes approximately:
#
# [0.63, 0.23, 0.14]
#
# These tell us how much attention each token receives.

attention_weights = F.softmax(
    attention_scores,
    dim=-1
)

print(attention_weights[0])

tensor([[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.1554, 0.8446, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.2384, 0.3894, 0.3722,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0121, 0.0141, 0.0077,  ..., 0.0071, 0.0000, 0.0000],
        [0.0375, 0.0596, 0.0187,  ..., 0.0111, 0.0694, 0.0000],
        [0.0137, 0.0331, 0.0429,  ..., 0.0484, 0.0229, 0.0430]],
       grad_fn=<SelectBackward0>)


In [ ]:
# Now use the attention weights
# to combine information from the Value vectors.

attention_output = attention_weights @ V

print(attention_output.shape)

torch.Size([16, 32, 16])


In [ ]:
N_HEAD = 4

class Head(nn.Module):

    def __init__(self, n_embd, head_size, block_size):
        super().__init__()

        # Create Query, Key and Value projections
        self.key = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        # Causal mask: prevents looking at future tokens
        self.register_buffer(
            "mask",
            torch.tril(
                torch.ones(block_size, block_size)
            )
        )

    def forward(self, x):

        B, T, C = x.shape

        # Create Key, Query and Value
        K = self.key(x)
        Q = self.query(x)
        V = self.value(x)

        # Attention scores
        attention_scores = Q @ K.transpose(-2, -1)

        # Scale
        attention_scores = attention_scores / (K.shape[-1] ** 0.5)

        # Prevent looking at future tokens
        attention_scores = attention_scores.masked_fill(
            self.mask[:T, :T] == 0,
            float("-inf")
        )

        # Convert scores to probabilities
        attention_weights = F.softmax(
            attention_scores,
            dim=-1
        )

        # Weighted combination of Values
        output = attention_weights @ V

        return output

class MultiHeadAttention(nn.Module):

    def __init__(self):
        super().__init__()

        head_size = N_EMBD // N_HEAD

        # Create 4 independent attention heads.
        self.heads = nn.ModuleList([
            Head(N_EMBD, head_size, block_size)
            for _ in range(N_HEAD)
        ])

        # Combine all heads back into 64 dimensions.
        self.projection = nn.Linear(
            N_EMBD,
            N_EMBD
        )

    def forward(self, x):

        # Each head looks at the sentence differently.
        outputs = [
            head(x)
            for head in self.heads
        ]

        # Join all heads together.
        x = torch.cat(
            outputs,
            dim=-1
        )

        # Mix the information from all heads.
        return self.projection(x)

In [ ]:
class FeedForward(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            # Expand:
            # 64 → 256
            nn.Linear(
                N_EMBD,
                4 * N_EMBD
            ),

            # Non-linearity.
            # Allows the network to learn
            # more complicated patterns.
            nn.GELU(),

            # Compress:
            # 256 → 64
            nn.Linear(
                4 * N_EMBD,
                N_EMBD
            )
        )

    def forward(self, x):

        return self.network(x)

In [ ]:
class Block(nn.Module):

    def __init__(self):
        super().__init__()

        self.attention = MultiHeadAttention()

        self.feed_forward = FeedForward()

        self.ln1 = nn.LayerNorm(
            N_EMBD
        )

        self.ln2 = nn.LayerNorm(
            N_EMBD
        )

    def forward(self, x):

        # LayerNorm
        # ↓
        # Attention
        # ↓
        # Residual connection
        x = x + self.attention(
            self.ln1(x)
        )

        # LayerNorm
        # ↓
        # Feed Forward
        # ↓
        # Residual connection
        x = x + self.feed_forward(
            self.ln2(x)
        )

        return x

In [ ]:
class TinyGPT(nn.Module):

    def __init__(self):

        super().__init__()

        # --------------------------------
        # TOKEN EMBEDDING
        # --------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            N_EMBD
        )

        # --------------------------------
        # POSITION EMBEDDING
        # --------------------------------

        self.position_embedding = nn.Embedding(
            block_size,
            N_EMBD
        )

        # --------------------------------
        # TRANSFORMER BLOCKS
        # --------------------------------

        self.blocks = nn.Sequential(
            Block(),
            Block(),
            Block(),
            Block()
        )

        # --------------------------------
        # FINAL NORMALIZATION
        # --------------------------------

        self.final_layer_norm = nn.LayerNorm(
            N_EMBD
        )

        # --------------------------------
        # FINAL LINEAR LAYER
        # --------------------------------

        # Convert 64-dimensional representation
        # into one score for every vocabulary token.

        self.lm_head = nn.Linear(
            N_EMBD,
            vocab_size
        )

    def forward(self, idx):

        B, T = idx.shape

        # Token embeddings
        token_vectors = self.token_embedding(idx)

        # Position embeddings
        positions = torch.arange(
            T,
            device=idx.device
        )

        position_vectors = self.position_embedding(
            positions
        )

        # Combine token + position
        x = token_vectors + position_vectors

        # Transformer
        x = self.blocks(x)

        # Final LayerNorm
        x = self.final_layer_norm(x)

        # Vocabulary scores
        logits = self.lm_head(x)

        return logits

In [ ]:
model = TinyGPT().to(device)

# Count how many learnable numbers exist.
number_of_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "Number of parameters:",
    number_of_parameters
)

Number of parameters: 204569


In [ ]:
X, Y = get_batch()

logits = model(X)

print("Logits shape:")
print(logits.shape)

Logits shape:
torch.Size([16, 32, 25])


In [ ]:
# Convert:
#
# [batch, time, vocabulary]
#
# into:
#
# [batch*time, vocabulary]

B, T, C = logits.shape

logits = logits.view(
    B * T,
    C
)

targets = Y.view(
    B * T
)


# Cross entropy asks:
#
# "How wrong was the model?"

loss = F.cross_entropy(
    logits,
    targets
)

print("Loss:", loss.item())

Loss: 3.3388495445251465


In [28]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)


for step in range(3000):

    # Get training examples.
    X, Y = get_batch()

    # --------------------------------
    # FORWARD PASS
    # --------------------------------

    logits = model(X)

    B, T, C = logits.shape

    loss = F.cross_entropy(
        logits.view(B * T, C),
        Y.view(B * T)
    )

    # --------------------------------
    # REMOVE OLD GRADIENTS
    # --------------------------------

    optimizer.zero_grad()

    # --------------------------------
    # BACKPROPAGATION
    # --------------------------------

    loss.backward()

    # --------------------------------
    # UPDATE PARAMETERS
    # --------------------------------

    optimizer.step()

    if step % 300 == 0:

        print(
            f"Step {step}, Loss: {loss.item():.4f}"
        )

Step 0, Loss: 3.3941
Step 300, Loss: 0.1027
Step 600, Loss: 0.0558
Step 900, Loss: 0.0375
Step 1200, Loss: 0.0342
Step 1500, Loss: 0.0382
Step 1800, Loss: 0.0464
Step 2100, Loss: 0.0507
Step 2400, Loss: 0.0290
Step 2700, Loss: 0.0414


KeyboardInterrupt: 

In [30]:
@torch.no_grad()
def generate(
    model,
    start_text,
    max_new_tokens=100
):

    # Convert starting text → token IDs.
    tokens = torch.tensor(
        [encode(start_text)],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        # Only give the model the last block_size tokens.
        context = tokens[:, -block_size:]

        # Get model predictions.
        logits = model(context)

        # Only care about the final token.
        logits = logits[:, -1, :]

        # Convert scores → probabilities.
        probabilities = F.softmax(
            logits,
            dim=-1
        )

        # Pick the next token.
        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        # Add it to our sequence.
        tokens = torch.cat(
            [tokens, next_token],
            dim=1
        )

    # Numbers → text.
    return decode(
        tokens[0].tolist()
    )

In [32]:
print(
    generate(
        model,
        "who is irfan",
        100
    )
)

who is irfan a good born in 2005. he like ice cream
eam
eorfan in is 205. crfan was born in 2005. he like ice cr
